In [1]:
# ============================================================
# MNIST CNN MODEL - SAVE, RELOAD, TFLITE AND FLASK API
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import time
import os

# ============================================================
# STEP 1: LOAD AND PREPROCESS MNIST DATASET
# ============================================================

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

# Reshape and normalize images
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)


# ============================================================
# STEP 2: BUILD CNN MODEL
# ============================================================

model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),

    layers.Flatten(),

    layers.Dense(64, activation='relu'),

    layers.Dense(10, activation='softmax')
])

# Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


# ============================================================
# STEP 3: TRAIN CNN MODEL
# ============================================================

history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)


# ============================================================
# STEP 4: EVALUATE ORIGINAL MODEL
# ============================================================

test_loss, test_acc = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\nOriginal Model Performance")
print("Test Loss:", test_loss)
print(f"Test Accuracy: {test_acc * 100:.2f}%")


# ============================================================
# STEP 5: SAVE MODEL
# ============================================================

os.makedirs('saved_model', exist_ok=True)

SAVE_PATH = 'saved_model/mnist_cnn.keras'

model.save(SAVE_PATH)

print("\nModel saved successfully!")
print("Model path:", SAVE_PATH)


# ============================================================
# STEP 6: RELOAD AND VERIFY SAVED MODEL
# ============================================================

loaded_model = tf.keras.models.load_model(SAVE_PATH)

reloaded_loss, reloaded_acc = loaded_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\nReloaded Model Performance")
print(f"Reloaded Model Accuracy: {reloaded_acc * 100:.2f}%")


# ============================================================
# STEP 7: CONVERT MODEL TO TENSORFLOW LITE
# ============================================================

converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

TFLITE_PATH = 'saved_model/mnist_cnn.tflite'

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_kb = os.path.getsize(TFLITE_PATH) / 1024

print("\nTensorFlow Lite Model Saved Successfully!")
print(f"TFLite Model Size: {size_kb:.2f} KB")


# ============================================================
# STEP 8: RUN TFLITE INFERENCE
# ============================================================

interpreter = tf.lite.Interpreter(
    model_path=TFLITE_PATH
)

interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

inp_idx = input_details[0]['index']
out_idx = output_details[0]['index']

# Select one test image
sample = X_test[0:1]

# Run inference
interpreter.set_tensor(inp_idx, sample)

interpreter.invoke()

# Get prediction
tflite_output = interpreter.get_tensor(out_idx)

tflite_pred = np.argmax(tflite_output)

# Original model prediction
original_output = model.predict(sample, verbose=0)

original_pred = np.argmax(original_output)

print("\nTFLite Inference Results")

print("Original Model Prediction:", original_pred)
print("TFLite Model Prediction:", tflite_pred)
print("Actual Label:", y_test[0])


# ============================================================
# STEP 9: CREATE FLASK REST API
# ============================================================

FLASK_APP = '''
from flask import Flask, request, jsonify
import tensorflow as tf
import numpy as np
import time

app = Flask(__name__)

# Load trained model
model = tf.keras.models.load_model(
    'saved_model/mnist_cnn.keras'
)

@app.route('/health', methods=['GET'])
def health():

    return jsonify({
        'status': 'ok',
        'model': 'mnist_cnn'
    })


@app.route('/predict', methods=['POST'])
def predict():

    data = request.get_json()

    img = np.array(
        data['image'],
        dtype='float32'
    ).reshape(1, 28, 28, 1) / 255.0

    start = time.time()

    probs = model.predict(img, verbose=0)[0]

    elapsed = (time.time() - start) * 1000

    return jsonify({

        'prediction': int(np.argmax(probs)),

        'confidence': float(np.max(probs)),

        'all_probs': [
            round(float(p), 4) for p in probs
        ],

        'inference_ms': round(elapsed, 2)

    })


if __name__ == '__main__':

    app.run(debug=True, port=5000)
'''

# Save Flask application
with open('app.py', 'w') as f:
    f.write(FLASK_APP.strip())

print("\nFlask application created successfully!")
print("Run using: python app.py")


# ============================================================
# STEP 10: SIMULATE API RESPONSE
# ============================================================

probs = model.predict(
    X_test[0:1],
    verbose=0
)[0]

simulated_response = {

    'prediction': int(np.argmax(probs)),

    'confidence': round(float(np.max(probs)), 4),

    'all_probs': [
        round(float(p), 4) for p in probs
    ],

    'inference_ms': 8.34

}

print("\nSimulated API Response:")

for k, v in simulated_response.items():

    print(f"{k}: {v}")


# ============================================================
# STEP 11: FINAL SUMMARY
# ============================================================

print("\n========== FINAL SUMMARY ==========")

print(f"Original Accuracy: {test_acc * 100:.2f}%")

print(f"Reloaded Accuracy: {reloaded_acc * 100:.2f}%")

print(f"TFLite Model Size: {size_kb:.2f} KB")

print("Keras Model: Saved Successfully")

print("TFLite Model: Saved Successfully")

print("Flask API: Created Successfully")

print("===================================")

Training data shape: (60000, 28, 28, 1)
Testing data shape: (10000, 28, 28, 1)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 26, 26, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 13, 13, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 11, 11, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 7744)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │         495,680 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │             650 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 515,146 (1.97 MB)

 Trainable params: 515,146 (1.97 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 24s 24ms/step - accuracy: 0.9543 - loss: 0.1533 - val_accuracy: 0.9835 - val_loss: 0.0573
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 20s 24ms/step - accuracy: 0.9850 - loss: 0.0482 - val_accuracy: 0.9892 - val_loss: 0.0409
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 18s 21ms/step - accuracy: 0.9900 - loss: 0.0314 - val_accuracy: 0.9898 - val_loss: 0.0356
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 17s 20ms/step - accuracy: 0.9929 - loss: 0.0218 - val_accuracy: 0.9905 - val_loss: 0.0360
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 21s 24ms/step - accuracy: 0.9951 - loss: 0.0157 - val_accuracy: 0.9893 - val_loss: 0.0385

Original Model Performance
Test Loss: 0.0348946638405323
Test Accuracy: 98.97%

Model saved successfully!
Model path: saved_model/mnist_cnn.keras

Reloaded Model Performance
Reloaded Model Accuracy: 98.97%
INFO:tensorflow:Assets written to: C:\Users\ADMIN\AppData\Local\Temp\tmpdvzojc3g\assets


INFO:tensorflow:Assets written to: C:\Users\ADMIN\AppData\Local\Temp\tmpdvzojc3g\assets


Saved artifact at 'C:\Users\ADMIN\AppData\Local\Temp\tmpdvzojc3g'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  3036440796560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3036440798288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3036440797904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3036440798672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3036440797136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3036440797520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3036440799248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3036440799440: TensorSpec(shape=(), dtype=tf.resource, name=None)

TensorFlow Lite Model Saved Successfully!
TFLite Model Size: 2016.38 KB


C:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



TFLite Inference Results
Original Model Prediction: 7
TFLite Model Prediction: 7
Actual Label: 7

Flask application created successfully!
Run using: python app.py

Simulated API Response:
prediction: 7
confidence: 1.0
all_probs: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0]
inference_ms: 8.34

========== FINAL SUMMARY ==========
Original Accuracy: 98.97%
Reloaded Accuracy: 98.97%
TFLite Model Size: 2016.38 KB
Keras Model: Saved Successfully
TFLite Model: Saved Successfully
Flask API: Created Successfully
